# Phase 5 - Statute identification with Arch 3 - Llama-style decoder

Starts from the Phase-1 model (12-layer Llama-style decoder (100.1M): RMSNorm, RoPE, grouped-query attention, SwiGLU, byte-level BPE.) and predicts **every** applicable statute (100 possible IDs, ~3.8 per
document) - a multi-label task scored by macro-F1, so rare statutes count as much as common ones. Statute mentions in
the text are masked (`<SECTION>`, `<ACT>`), so they have to be inferred from the facts.

**The model**
* Up to 6 chunks of 512 tokens per document, spread evenly from its start to its end - statutes can be implied anywhere.
  The Phase-1 backbone encodes each chunk (causal mask lifted for the decoder-only models) and every token is tagged
  with the chunk it came from.
* **Label-wise attention** (CAML, Mullenbach et al. 2018; LWAN for legal text, Chalkidis et al. 2019): each statute has
  its own attention over all tokens of the document and its own scorer, so the evidence for statute 34 is collected
  separately from the evidence for statute 5.
* **Asymmetric loss** (Ridnik et al. 2021) for the heavy positive/negative imbalance (4 positives vs 96 negatives).
* **Per-statute thresholds** tuned on the dev set (macro-F1 is the average of per-statute F1s); statutes too rare on
  dev fall back to a global threshold, and every document gets at least one statute.

**Experiments** (each is its own submission; all from the Phase-5 list)

| Experiment | What trains |
|---|---|
| `full` | every weight, layer-wise LR decay - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `adapter` | Houlsby bottleneck adapters after the attention and feed-forward outputs (Houlsby et al. 2019) |
| `partial` | only the top backbone layers + final norm (layer-wise / partial fine-tuning) |
| `prompt` | prompt tuning: 16 learned virtual tokens in front of every chunk (Lester et al. 2021) |
| `kd` | distillation: the fine-tuned `full` model teaches a half-depth student through its per-statute probabilities and its document representation (Hinton et al. 2015) |

The label-wise attention head trains in every experiment. 42K training documents are more than one session can
read many times over, so each experiment trains for a fixed **time budget** (`TRAIN_MINUTES`).

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [1]:
import os, sys, json, glob, shutil, time, gc
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase5_arch3"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPUs: ['Tesla T4', 'Tesla T4']


## 2. Configuration
Everything you might want to change lives here.

In [2]:
%%writefile /kaggle/working/phase5_arch3/config.py
"""Every knob of this Phase-5 (legal statute identification) notebook."""
import os
import json

ARCH = "arch3"
WORK_DIR = f"/kaggle/working/phase5_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
RAW = {s: os.path.join(DATA_DIR, "Phase_5_LSI", "Test" if s == "test" else "Train", f"{s}.json")
       for s in ("train", "dev", "test")}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch3", then these HF repos.
PHASE1_REPOS = ["D-240357/legal-arch3-decoder"]
WEIGHT_FILES = ["model_final.pt", "model_best.pt"]
TOKENIZER_FILES = ["tokenizer.json", "data/tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS
KAGGLE_HINT = "arch3"

LSI_REPO = "D-240357/legal-lsi-arch3"           # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish.
METHODS = ["full", "lorma", "adapter", "partial", "prompt", "kd"]
KD_TEACHER = "full"
TRAIN_MINUTES = 40             # wall-clock training budget per experiment (the LR schedule ends exactly on it)
MAX_EPOCHS = 3

# ------------------------------------------------------------------ how a document is read
CHUNK_TOKENS = 512
MAX_CHUNKS = 6                 # long documents are sampled at 6 evenly spaced places, first and last included
CHAR_CAP = 150_000             # text beyond this is not even tokenized (99% of documents are < 50K characters)
BIDIRECTIONAL = True            # lift the causal mask while fine-tuning (Dukic & Snajder 2024; LLM2Vec)
LWAN_DIM = 256                 # label-wise attention size
HEAD_DROPOUT = 0.1

# ------------------------------------------------------------------ loss and thresholds
LOSS = "asl"                   # "asl" (asymmetric loss, Ridnik et al. 2021) or "bce"
ASL_GAMMA_NEG = 2.0
ASL_GAMMA_POS = 0.0
ASL_CLIP = 0.05
THRESHOLDS = "auto"            # "per_label", "global" or "auto" (whichever is better on dev)
MIN_DEV_POSITIVES = 20         # statutes rarer than this on dev use the global threshold

# ------------------------------------------------------------------ optimisation
MICRO_DOCS = 3                 # documents per forward pass (each document = up to 6 chunks)
GRAD_ACCUM = 6
EVAL_BATCH = 12
WARMUP_STEPS = 50
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
HEAD_LR = 1e-3
FULL_LR = 5e-5                 # top layer; each layer below gets LLRD times the one above it
LLRD = 0.9
PARTIAL_LR = 1e-4
PARTIAL_TOP_LAYERS = 4
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["qkv", "proj"]       # LoRMA: attention projections
ADAPTER_SITES = ["proj", "down"]           # Houlsby adapters: after the attention output and the feed-forward output
ADAPTER_BOTTLENECK = 64
PROMPT_LEN = 16
PROMPT_LR = 1e-2

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the gold-label loss; the rest goes to matching the teacher
KD_HIDDEN_WEIGHT = 0.5         # also align the student's document representation with the teacher's (cosine)
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

# ------------------------------------------------------------------ evaluation
EVAL_EVERY_STEPS = 150         # quick check on DEV_SUBSET documents; the best checkpoint is kept
DEV_SUBSET = 1500
DEV_FINAL_LIMIT = None         # thresholds are tuned on the whole dev set (10,181 documents)
TRAIN_DOC_LIMIT = None         # set to e.g. 400 (and TRAIN_MINUTES = 3) for a quick smoke test
PREP_WORKERS = max(1, os.cpu_count() or 2)

SEED = 13

_override = os.environ.get("P5_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

Writing /kaggle/working/phase5_arch3/config.py


## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the statute-identification pieces.

In [3]:
%%writefile /kaggle/working/phase5_arch3/tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])


# ---------------------------------------------------------------------------------------------------
# Phase 2 addition: encode with character offsets, so predicted tokens can be mapped back to exact spans.
# ---------------------------------------------------------------------------------------------------
# Same splitter as above, except a run of punctuation is cut into single characters. Gold entities often end
# in the middle of runs like "Ors.[1]" or "Cr.P.C.,"; with this split 99.3% of gold spans start and end on a
# token boundary (98.1% without it). Every piece is still encoded with the Phase-1 merges.
NER_PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]|\s+(?!\S)|\s+",
    re.UNICODE,
)


class NerTokenizer(ByteBPE):
    split_punct = True

    def encode_with_offsets(self, text):
        """Returns ids and, per token, the [start, end) of its non-whitespace characters in `text`."""
        splitter = NER_PRETOKEN_RE if self.split_punct else PRETOKEN_RE
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            word, base = m.group(), m.start()
            toks = self._cache.get(word)
            if toks is None:
                toks = self._cache[word] = self._encode_word(word)
            # byte position -> character position inside this piece (pieces can contain multi-byte characters)
            char_at_byte = []
            for ci, ch in enumerate(word):
                char_at_byte += [ci] * len(ch.encode("utf-8"))
            b = 0
            for t in toks:
                n = len(self.vocab[t])
                cs, ce = char_at_byte[b], char_at_byte[b + n - 1] + 1
                b += n
                piece = word[cs:ce]
                s = base + cs + (len(piece) - len(piece.lstrip()))
                e = base + ce - (len(piece) - len(piece.rstrip()))
                if s >= e:
                    s = e = base + ce
                ids.append(t)
                starts.append(s)
                ends.append(e)
        return ids, starts, ends

Writing /kaggle/working/phase5_arch3/tokenizer.py


In [4]:
%%writefile /kaggle/working/phase5_arch3/phase1_model.py
"""
Phase-1 model code, unchanged except for the small switches marked "Phase 2", so the saved weights load as-is.
"""
"""
Arch 3: a compact Llama-style decoder, written from scratch in plain PyTorch ops.

  * RMSNorm (pre-norm), no biases anywhere
  * Rotary position embeddings (RoPE) instead of a learned position table
  * Grouped-query attention: fewer KV heads than query heads -> smaller KV cache, fewer params
  * Attention goes through F.scaled_dot_product_attention, so the T x T score matrix is never
    materialised (fused flash / memory-efficient kernel is picked automatically)
  * SwiGLU feed-forward
  * Tied input/output embeddings
  * KV-cache generation, batched, with left-padded prompts
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    d_model: int = 768
    d_ff: int = 2048
    max_seq_len: int = 2048
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    z_loss: float = 1e-4


class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # statistics in fp32 even when the surrounding math is fp16
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(head_dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)       # (T, head_dim/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x: (B, heads, T, head_dim); rotate the two halves of every head vector
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv = cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.group = cfg.n_head // cfg.n_kv_head
        q_dim = self.n_head * self.head_dim
        kv_dim = self.n_kv * self.head_dim
        self.qkv = nn.Linear(cfg.d_model, q_dim + 2 * kv_dim, bias=False)   # one fused matmul
        self.proj = nn.Linear(q_dim, cfg.d_model, bias=False)
        self.q_dim, self.kv_dim = q_dim, kv_dim
        self.causal = True                         # Phase 2 turns this off to let tokens look right

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split([self.q_dim, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv, self.head_dim).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past is not None:                       # decoding: stash un-expanded K/V for reuse
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v)

        if self.group > 1:                         # share each KV head across `group` query heads
            k = k.repeat_interleave(self.group, dim=1)
            v = v.repeat_interleave(self.group, dim=1)

        if key_mask is None:
            # training path: plain causal attention, fastest kernel
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)

        out = out.transpose(1, 2).reshape(B, T, self.q_dim)
        return self.proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_up = nn.Linear(cfg.d_model, 2 * cfg.d_ff, bias=False)     # fused gate + up
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        a, present = self.attn(self.attn_norm(x), cos, sin, past, key_mask)
        x = x + a
        x = x + self.ffn(self.ffn_norm(x))
        return x, present


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_head, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 trick: shrink the layers that write into the residual stream so the
        # variance does not grow with depth.
        scale = 0.02 / math.sqrt(2 * cfg.n_layer)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=scale)
            nn.init.normal_(blk.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())        # tied weights are counted once

    def hidden_states(self, idx):
        """Final-layer hidden states (B, T, d_model) - what a Phase 2-7 task head sits on top of."""
        B, T = idx.shape
        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x, _ = blk(x, cos, sin)
        return self.final_norm(x)

    def forward(self, idx, targets=None, caches=None, start_pos=0, key_mask=None, last_only=False):
        """
        Training:   model(idx, targets)            -> (ce_loss, z_loss)
        Inference:  model(idx, caches=..., ...)    -> (logits, new_caches)
        """
        B, T = idx.shape
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        x = self.tok_emb(idx)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            past = caches[i] if caches is not None else None
            x, present = blk(x, cos, sin, past, key_mask)
            new_caches.append(present)
        x = self.final_norm(x)
        if last_only:
            x = x[:, -1:, :]

        logits = F.linear(x, self.tok_emb.weight)               # tied head

        if targets is None:
            return logits, new_caches

        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()
        return ce, z


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    """logits: (B, V) fp32 -> next token ids (B,)"""
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)                   # already sorted, descending
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs                       # mass strictly before each token
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    choice = torch.multinomial(probs, 1)
    return idx.gather(-1, choice).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=384, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """
    prompts: list of token-id lists (any lengths). Returns a list of generated strings
    (the continuation only, trimmed to `target_words` words).
    Prompts are left-padded to a common length; padded keys are masked out, and because RoPE
    is relative, the shift does not change what the real tokens see.
    """
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    L = max(len(p) for p in prompts)
    assert L + max_new_tokens <= model.cfg.max_seq_len, "prompt + continuation exceeds context"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, L), pad, dtype=torch.long)
    valid = torch.zeros((B, L), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, L - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, L - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    # prefill: causal mask AND key-validity; diagonal always allowed so pad rows never go all -inf
    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None, None]
    logits, caches = model(ids, caches=None, start_pos=0, key_mask=mask, last_only=True)

    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    for step in range(max_new_tokens):
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:                          # the task wants a full-length continuation: stop on word count only
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

        valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=dev)], dim=1)
        step_mask = valid[:, None, None, :]
        logits, caches = model(nxt[:, None], caches=caches, start_pos=L + step,
                               key_mask=step_mask, last_only=True)

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

Writing /kaggle/working/phase5_arch3/phase1_model.py


In [5]:
%%writefile /kaggle/working/phase5_arch3/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

Writing /kaggle/working/phase5_arch3/hub_files.py


In [6]:
%%writefile /kaggle/working/phase5_arch3/backbone.py
"""Loads the Phase-1 arch 3 weights (Llama-style decoder) and exposes them as a sentence encoder for rhetorical roles."""
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_rr)
from phase1_model import GPT, GPTConfig
from tokenizer import NerTokenizer


def add_prompt(x, mask, prompt):
    """Prompt tuning (Lester et al. 2021): prepend learned vectors to the token embeddings."""
    if prompt is None:
        return x, mask, 0
    P = prompt.size(0)
    x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
    mask = torch.cat([mask.new_ones(mask.size(0), P), mask], dim=1)
    return x, mask, P


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer.load(path)
    tok.sep_id = tok.eos_id               # marks the start of every sentence inside a pack
    return tok


class Backbone(nn.Module):
    def __init__(self, gpt, keep_layers, bidirectional):
        super().__init__()
        self.d_model = gpt.cfg.d_model
        self.tok_emb = gpt.tok_emb
        blocks = gpt.blocks if keep_layers is None else [gpt.blocks[i] for i in keep_layers]
        self.layers = nn.ModuleList(blocks)
        self.final_norm = gpt.final_norm
        self.register_buffer("rope_cos", gpt.rope_cos.clone(), persistent=False)
        self.register_buffer("rope_sin", gpt.rope_sin.clone(), persistent=False)
        self.bidirectional = bidirectional
        for blk in self.layers:
            blk.attn.causal = not bidirectional

    def forward(self, ids, mask, prompt=None):
        x, mask, P = add_prompt(self.tok_emb(ids), mask, prompt)
        T = x.size(1)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = mask[:, None, None, :] if (self.bidirectional and not bool(mask.all())) else None
        for blk in self.layers:
            x, _ = blk(x, cos, sin, None, key_mask)
        return self.final_norm(x)[:, P:]


def build_backbone(keep_layers=None, bidirectional=True):
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    blob = torch.load(path, map_location="cpu")
    gpt = GPT(GPTConfig(**blob["model_cfg"]))
    gpt.load_state_dict({k: v.float() for k, v in blob["model"].items()})
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 3 weights: {path} ({source})", flush=True)
    return Backbone(gpt, keep_layers, bidirectional)


def student_layers():
    return list(C.STUDENT_LAYERS)

Writing /kaggle/working/phase5_arch3/backbone.py


In [7]:
%%writefile /kaggle/working/phase5_arch3/lsi_core.py
"""
Phase 5 (legal statute identification) machinery shared by all four architectures.

A document becomes up to MAX_CHUNKS chunks of 512 tokens spread evenly from its start to its end (statutes can be
implied anywhere in the facts). The Phase-1 backbone encodes every chunk; each token is tagged with the chunk it came
from; then label-wise attention (CAML, Mullenbach et al. 2018; LWAN for legal text, Chalkidis et al. 2019) lets every
statute attend to the tokens that matter for *it* before its own score is computed.
"""
import os
import json
import math
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

N_LABELS = 100


# ------------------------------------------------------------------------------------------- data
def load_docs(path, with_labels):
    with open(path, encoding="utf-8") as f:
        rows = json.load(f)
    ids = [r["id"] for r in rows]
    texts = [r["text"] if isinstance(r["text"], str) else " ".join(r["text"]) for r in rows]
    labels = [sorted(set(int(l) for l in r["labels"])) for r in rows] if with_labels else None
    return ids, texts, labels


_tok = None
_cfg = None


def _init_worker(cfg_dict):
    global _tok, _cfg
    import backbone as BB
    _tok = BB.load_tokenizer()
    _cfg = cfg_dict


def chunk_text(text):
    """Up to K chunks of W tokens; long documents are sampled at K evenly spaced places (first and last included)."""
    W, K = _cfg["W"], _cfg["K"]
    ids = _tok.encode(text[:_cfg["char_cap"]]) or [_cfg["pad"]]      # an empty document still gets one token
    n = len(ids)
    if n <= K * W:
        starts = list(range(0, n, W)) or [0]
    else:
        starts = [round(i * (n - W) / (K - 1)) for i in range(K)]
    out = np.full((K, W), _cfg["pad"], dtype=np.int32)
    lens = np.zeros(K, dtype=np.int16)
    for k, s in enumerate(starts[:K]):
        piece = ids[s:s + W]
        out[k, :len(piece)] = piece
        lens[k] = len(piece)
    return out, lens


def prepare_split(texts, cfg_dict, workers):
    with Pool(workers, initializer=_init_worker, initargs=(cfg_dict,)) as pool:
        res = pool.map(chunk_text, texts, chunksize=64)
    return np.stack([r[0] for r in res]), np.stack([r[1] for r in res])


def multi_hot(labels):
    y = np.zeros((len(labels), N_LABELS), dtype=np.float32)
    for i, ls in enumerate(labels):
        y[i, ls] = 1.0
    return y


# ----------------------------------------------------------------------------------- adaptation
class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class Adapter(nn.Module):
    """Houlsby et al. 2019 bottleneck adapter on a sub-layer's output: h + up(gelu(down(h))). up = 0 at the start."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.up(F.gelu(self.down(h)))


def wrap_linears(root, names, make, skip=("router",)):
    """MoE routers are always left alone: arch 4 reads router.weight directly."""
    n = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in skip:
                continue
            if (names is None or name in names) and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                n += 1
    return n


# ---------------------------------------------------------------------------------------- model
class LSIModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        D = backbone.d_model
        self.chunk_pos = nn.Embedding(cfg.MAX_CHUNKS, D)
        nn.init.zeros_(self.chunk_pos.weight)
        self.drop = nn.Dropout(cfg.HEAD_DROPOUT)
        self.proj = nn.Linear(D, cfg.LWAN_DIM)
        self.label_query = nn.Parameter(torch.randn(N_LABELS, cfg.LWAN_DIM) * 0.02)   # one attention per statute
        self.label_out = nn.Parameter(torch.randn(N_LABELS, cfg.LWAN_DIM) * 0.02)     # one scorer per statute
        self.label_bias = nn.Parameter(torch.zeros(N_LABELS))
        self.prompt = None

    def token_states(self, ids, lens):
        """ids (B, K, W) -> token states (B, K*W, D) and mask; empty chunks never reach the backbone."""
        B, K, W = ids.shape
        flat_ids, flat_lens = ids.view(B * K, W), lens.view(B * K)
        real = flat_lens > 0
        L = int(flat_lens.max())
        states = torch.zeros(B * K, L, self.backbone.d_model, device=ids.device)
        sub_ids = flat_ids[real][:, :L]
        mask = torch.arange(L, device=ids.device)[None, :] < flat_lens[real][:, None]
        h = self.backbone(sub_ids, mask, prompt=self.prompt).float()
        states = states.index_put((real.nonzero().squeeze(1),), h)
        full_mask = torch.arange(L, device=ids.device)[None, :] < flat_lens[:, None]
        states = states.view(B, K, L, -1) + self.chunk_pos.weight[None, :, None, :]
        return states.view(B, K * L, -1), full_mask.view(B, K * L)

    def forward(self, ids, lens):
        h, mask = self.token_states(ids, lens)
        z = torch.tanh(self.proj(self.drop(h)))                                  # (B, T, A)
        scores = torch.einsum("bta,la->blt", z, self.label_query)                # (B, L, T)
        scores = scores.masked_fill(~mask[:, None, :], float("-inf"))
        att = torch.softmax(scores, dim=-1)
        per_label = torch.einsum("blt,bta->bla", att, z)                         # (B, L, A)
        logits = (per_label * self.label_out[None]).sum(-1) + self.label_bias
        doc = (h * mask[..., None]).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        return logits.float(), doc


def configure_method(model, method, cfg):
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"
    for p in bb.parameters():
        p.requires_grad = False
    if method == "lorma":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "adapter":
        n = wrap_linears(bb.layers, cfg.ADAPTER_SITES, lambda m: Adapter(m, cfg.ADAPTER_BOTTLENECK))
        return f"{n} bottleneck adapters (size {cfg.ADAPTER_BOTTLENECK}) after attention and feed-forward outputs"
    if method == "partial":
        for layer in list(bb.layers)[-cfg.PARTIAL_TOP_LAYERS:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in bb.final_norm.parameters():
            p.requires_grad = True
        return f"top {cfg.PARTIAL_TOP_LAYERS} backbone layers + final norm trainable"
    if method == "prompt":
        # Lester et al. 2021: learned virtual tokens, initialised from real token embeddings
        g = torch.Generator().manual_seed(cfg.SEED)
        w = bb.tok_emb.weight
        pick = torch.randint(100, w.size(0) - 100, (cfg.PROMPT_LEN,), generator=g)
        model.prompt = nn.Parameter(w[pick].detach().clone())
        return f"{cfg.PROMPT_LEN} soft-prompt vectors prepended to every chunk"
    raise ValueError(method)


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def layer_depth(name, n_layers):
    if ".layers." in name:
        return int(name.split(".layers.")[1].split(".")[0]) + 1
    if "emb" in name and name.startswith("backbone."):
        return 0
    return n_layers + 1


def build_optimizer(model, method, cfg):
    n_layers = len(model.backbone.layers)
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name == "prompt":
            lr = cfg.PROMPT_LR
        elif not name.startswith("backbone."):
            lr = cfg.HEAD_LR
        elif method in ("full", "kd"):
            lr = cfg.FULL_LR * cfg.LLRD ** (n_layers + 1 - layer_depth(name, n_layers))
        elif method == "partial":
            lr = cfg.PARTIAL_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if (p.ndim < 2 or name == "prompt") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# ----------------------------------------------------------------------------------------- loss
def asymmetric_loss(logits, y, gamma_neg, gamma_pos, clip):
    """Ridnik et al. 2021 (ASL): easy negatives are down-weighted and very-low-probability negatives ignored,
    which suits 100 statutes where each document has ~4 positives."""
    p = torch.sigmoid(logits)
    p_neg = (p - clip).clamp(min=0) if clip > 0 else p
    loss_pos = y * torch.log(p.clamp(min=1e-8)) * (1 - p) ** gamma_pos
    loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=1e-8)) * p_neg ** gamma_neg
    return -(loss_pos + loss_neg).sum(1).mean()


def task_loss(logits, y, cfg):
    if cfg.LOSS == "asl":
        return asymmetric_loss(logits, y, cfg.ASL_GAMMA_NEG, cfg.ASL_GAMMA_POS, cfg.ASL_CLIP)
    return F.binary_cross_entropy_with_logits(logits, y, reduction="sum") / logits.size(0)


# -------------------------------------------------------------------------------------- metrics
def scores(Y, P):
    """Macro-F1 (over the 100 statutes), micro-F1 and subset accuracy for binary matrices Y (gold) and P (pred)."""
    tp = (Y * P).sum(0)
    fp = ((1 - Y) * P).sum(0)
    fn = (Y * (1 - P)).sum(0)
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    present = Y.sum(0) + P.sum(0) > 0
    micro = 2 * tp.sum() / max(1.0, 2 * tp.sum() + fp.sum() + fn.sum())
    return {"macro_f1": float(f1[present].mean()), "micro_f1": float(micro),
            "subset_acc": float((Y == P).all(1).mean()), "per_label_f1": f1.tolist()}


def decide(probs, thresholds, at_least_one=True):
    P = (probs >= thresholds[None, :]).astype(np.float32)
    if at_least_one:
        empty = P.sum(1) == 0
        P[empty, probs[empty].argmax(1)] = 1.0           # every training document has at least one statute
    return P


def tune_thresholds(Y, probs, min_pos):
    """Per-statute threshold that maximises that statute's dev F1 (macro-F1 is their average).
    Statutes with fewer than min_pos dev positives fall back to the best single global threshold."""
    grid = np.linspace(0.05, 0.95, 91)
    glob = max(grid, key=lambda t: scores(Y, decide(probs, np.full(N_LABELS, t)))["macro_f1"])
    th = np.full(N_LABELS, glob)
    for l in range(N_LABELS):
        if Y[:, l].sum() < min_pos:
            continue
        y, p = Y[:, l], probs[:, l]
        best, best_f1 = glob, -1.0
        for t in grid:
            pred = p >= t
            tp = (pred & (y == 1)).sum()
            f1 = 2 * tp / max(1, pred.sum() + y.sum())
            if f1 > best_f1:
                best, best_f1 = t, f1
        th[l] = best
    return float(glob), th


@torch.no_grad()
def predict_probs(model, ids, lens, device, batch):
    model.eval()
    out = []
    for b in range(0, len(ids), batch):
        x = torch.from_numpy(ids[b:b + batch].astype(np.int64)).to(device)
        l = torch.from_numpy(lens[b:b + batch].astype(np.int64)).to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits, _ = model(x, l)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)


def write_submission(test_ids, P, out_dir, zip_name):
    import zipfile
    assert len(test_ids) == len(P) == len(set(map(str, test_ids)))
    payload = [{"id": i, "pred": sorted(int(l) for l in np.nonzero(row)[0])} for i, row in zip(test_ids, P)]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w") as f:
        json.dump(payload, f)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path

Writing /kaggle/working/phase5_arch3/lsi_core.py


In [8]:
%%writefile /kaggle/working/phase5_arch3/run_lsi.py
"""
One Phase-5 experiment = one call:

    python run_lsi.py --method lorma

Trains for a fixed wall-clock budget, checks a dev subset along the way and keeps the best checkpoint, then tunes one
threshold per statute on the whole dev set, predicts the test documents and pushes everything to Hugging Face.
"""
import os
import sys
import json
import math
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from lsi_core import (LSIModel, configure_method, count_params, trainable_state, build_optimizer, task_loss, scores,
                      decide, tune_thresholds, predict_probs, write_submission, N_LABELS)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = LSIModel(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    path = os.path.join(C.OUT_DIR, method, f"lsi_{method}.pt")
    if not os.path.exists(path):
        path = BB.fetch_from_hub(C.LSI_REPO, f"{method}/lsi_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.LSI_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.LSI_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.LSI_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.LSI_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def load_npz(name):
    z = np.load(os.path.join(C.DATA_DIR, f"{name}.npz"))
    return z["ids"], z["lens"], (z["y"] if "y" in z.files else None)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lorma", "adapter", "partial", "prompt", "kd"])
    method = ap.parse_args().method
    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    tr_ids, tr_lens, tr_y = load_npz("train")
    dv_ids, dv_lens, dv_y = load_npz("dev")
    te_ids, te_lens, _ = load_npz("test")
    with open(os.path.join(C.DATA_DIR, "test_ids.json")) as f:
        test_doc_ids = json.load(f)
    if C.TRAIN_DOC_LIMIT:
        tr_ids, tr_lens, tr_y = tr_ids[:C.TRAIN_DOC_LIMIT], tr_lens[:C.TRAIN_DOC_LIMIT], tr_y[:C.TRAIN_DOC_LIMIT]
    if C.DEV_FINAL_LIMIT:
        dv_ids, dv_lens, dv_y = dv_ids[:C.DEV_FINAL_LIMIT], dv_lens[:C.DEV_FINAL_LIMIT], dv_y[:C.DEV_FINAL_LIMIT]
    sub = np.random.RandomState(0).permutation(len(dv_y))[:C.DEV_SUBSET]
    log(f"[{method}] train {len(tr_y):,} docs | dev {len(dv_y):,} | test {len(test_doc_ids):,} | "
        f"chunks used per doc {(tr_lens > 0).sum(1).mean():.2f} of {tr_ids.shape[1]} | statutes per doc {tr_y.sum(1).mean():.2f}")

    keep = BB.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M | loss {C.LOSS} | budget {C.TRAIN_MINUTES} min")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> "
            f"student with backbone layers {keep} ({total / 1e6:.1f}M)")

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    docs_per_step = C.MICRO_DOCS * C.GRAD_ACCUM
    step_cap = max(1, C.MAX_EPOCHS * len(tr_y) // docs_per_step)
    budget = C.TRAIN_MINUTES * 60
    order = np.random.RandomState(C.SEED).permutation(len(tr_y))
    cursor, step, seen, running = 0, 0, 0, 0.0
    best, history = {"f1": -1.0}, []
    t_train = time.time()

    def progress():
        return max(step / step_cap, (time.time() - t_train) / budget)

    while progress() < 1.0:
        p = progress()
        warm = min(1.0, (step + 1) / C.WARMUP_STEPS)
        lr_scale = warm * (C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))
        for g, lr in zip(optimizer.param_groups, base_lrs):
            g["lr"] = lr * lr_scale

        model.train()
        for _ in range(C.GRAD_ACCUM):
            if cursor + C.MICRO_DOCS > len(order):
                order = np.random.RandomState(C.SEED + step).permutation(len(tr_y))
                cursor = 0
            idx = np.sort(order[cursor:cursor + C.MICRO_DOCS])
            cursor += C.MICRO_DOCS
            x = torch.from_numpy(tr_ids[idx].astype(np.int64)).to(device)
            l = torch.from_numpy(tr_lens[idx].astype(np.int64)).to(device)
            y = torch.from_numpy(tr_y[idx]).to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits, doc = model(x, l)
                loss = task_loss(logits, y, C)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits, t_doc = teacher(x, l)
                    T = C.KD_TEMPERATURE
                    soft = torch.sigmoid(t_logits.float() / T)          # per-statute soft labels from the teacher
                    kd = F.binary_cross_entropy_with_logits(logits / T, soft, reduction="sum") / logits.size(0) * T * T
                    rep = (1 - F.cosine_similarity(doc.float(), t_doc.float(), dim=-1)).mean()
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kd + C.KD_HIDDEN_WEIGHT * rep
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([q for q in model.parameters() if q.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % C.EVAL_EVERY_STEPS == 0 or progress() >= 1.0:
            probs = predict_probs(model, dv_ids[sub], dv_lens[sub], device, C.EVAL_BATCH)
            glob, _ = tune_thresholds(dv_y[sub], probs, min_pos=10 ** 9)            # global threshold only, for speed
            s = scores(dv_y[sub], decide(probs, np.full(N_LABELS, glob)))
            history.append({"step": step, "docs_seen": step * docs_per_step, "train_loss": running / max(1, seen),
                            "dev_subset_macro_f1": s["macro_f1"], "dev_subset_micro_f1": s["micro_f1"],
                            "minutes": (time.time() - t0) / 60})
            log(f"[{method}] step {step} | {step * docs_per_step:,} docs seen | loss {running / max(1, seen):.4f} | "
                f"dev-subset macro-F1 {s['macro_f1']:.4f} micro-F1 {s['micro_f1']:.4f} (thr {glob:.2f}) | "
                f"{100 * min(1.0, progress()):.0f}% of budget")
            running, seen = 0.0, 0
            if s["macro_f1"] > best["f1"]:
                best = {"f1": s["macro_f1"], "step": step}
                torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model), "step": step},
                           os.path.join(out_dir, f"lsi_{method}.pt"))

    # ------------------------------- best checkpoint: per-statute thresholds on the full dev set, then test
    blob = torch.load(os.path.join(out_dir, f"lsi_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dev_probs = predict_probs(model, dv_ids, dv_lens, device, C.EVAL_BATCH)
    dev_seconds = time.time() - t_inf
    glob, per_label = tune_thresholds(dv_y, dev_probs, C.MIN_DEV_POSITIVES)
    s_glob = scores(dv_y, decide(dev_probs, np.full(N_LABELS, glob)))
    s_per = scores(dv_y, decide(dev_probs, per_label))
    use_per = C.THRESHOLDS == "per_label" or (C.THRESHOLDS == "auto" and s_per["macro_f1"] >= s_glob["macro_f1"])
    thresholds = per_label if use_per else np.full(N_LABELS, glob)
    final = s_per if use_per else s_glob

    test_probs = predict_probs(model, te_ids, te_lens, device, C.EVAL_BATCH)
    test_P = decide(test_probs, thresholds)
    _, zip_path = write_submission(test_doc_ids, test_P, out_dir, f"submission_{C.ARCH}_{method}.zip")
    np.save(os.path.join(out_dir, "dev_probs.npy"), dev_probs.astype(np.float16))

    support = dv_y.sum(0)
    worst = sorted(range(N_LABELS), key=lambda l: final["per_label_f1"][l])[:10]
    summary = {
        "arch": C.ARCH, "method": method, "best_step": best["step"],
        "dev_macro_f1": final["macro_f1"], "dev_micro_f1": final["micro_f1"], "dev_subset_acc": final["subset_acc"],
        "thresholds": "per_label" if use_per else "global", "global_threshold": glob,
        "dev_macro_f1_global_thr": s_glob["macro_f1"], "dev_macro_f1_per_label_thr": s_per["macro_f1"],
        "dev_per_label_f1": final["per_label_f1"], "dev_label_support": support.tolist(),
        "worst_labels": [{"label": l, "f1": final["per_label_f1"][l], "dev_support": int(support[l])} for l in worst],
        "test_labels_per_doc": float(test_P.sum(1).mean()),
        "params_total": total, "params_trainable": trainable, "dev_inference_seconds": round(dev_seconds, 1),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "minutes": round((time.time() - t0) / 60, 1), "docs_seen": step * docs_per_step, "history": history,
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | dev macro-F1 {final['macro_f1']:.4f} (global thr {s_glob['macro_f1']:.4f}, per-statute thr "
        f"{s_per['macro_f1']:.4f}) | micro-F1 {final['micro_f1']:.4f} | subset acc {final['subset_acc']:.4f} | "
        f"{test_P.sum(1).mean():.2f} statutes per test doc -> {zip_path}")

    push([(os.path.join(out_dir, f"lsi_{method}.pt"), f"{method}/lsi_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev macro-F1 {final['macro_f1']:.4f}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/phase5_arch3/run_lsi.py


## 4. Data: download and turn every document into chunks
Tokenized once with all CPU cores and stored as compact arrays, so each experiment starts instantly.

In [9]:
import importlib
import config as C
importlib.reload(C)
import backbone as BB
import lsi_core as L
importlib.reload(L)

for split in ("train", "dev", "test"):
    folder = "Test" if split == "test" else "Train"
    hf_hub_download(C.HF_DATASET_REPO, f"Phase_5_LSI/{folder}/{split}.json", repo_type="dataset",
                    local_dir=C.DATA_DIR, token=HF_TOKEN)

tok = BB.load_tokenizer()                       # caches the tokenizer before the worker processes start
chunk_cfg = {"W": C.CHUNK_TOKENS, "K": C.MAX_CHUNKS, "char_cap": C.CHAR_CAP, "pad": tok.pad_id}

for split in ("train", "dev", "test"):
    t0 = time.time()
    ids, texts, labels = L.load_docs(C.RAW[split], with_labels=split != "test")
    chunk_ids, chunk_lens = L.prepare_split(texts, chunk_cfg, C.PREP_WORKERS)
    arrays = {"ids": chunk_ids, "lens": chunk_lens}
    if labels is not None:
        arrays["y"] = L.multi_hot(labels)
        assert all(0 <= l < L.N_LABELS for ls in labels for l in ls)
    np.savez(os.path.join(C.DATA_DIR, f"{split}.npz"), **arrays)
    if split == "test":
        json.dump(ids, open(os.path.join(C.DATA_DIR, "test_ids.json"), "w"))
    extra = f" | statutes per doc {arrays['y'].sum(1).mean():.2f}" if labels is not None else ""
    print(f"{split}: {len(ids):,} docs, chunks per doc {(chunk_lens > 0).sum(1).mean():.2f} of {C.MAX_CHUNKS} "
          f"| {time.time() - t0:.0f}s{extra}")
    del texts, chunk_ids, chunk_lens, arrays
    gc.collect()

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/413k [00:00<?, ?B/s]

train: 42,750 docs, chunks per doc 2.72 of 6 | 25s | statutes per doc 3.70
dev: 10,181 docs, chunks per doc 2.93 of 6 | 7s | statutes per doc 3.92
test: 13,019 docs, chunks per doc 2.89 of 6 | 9s


## 5. Load the Phase-1 weights once
This checks the model end to end and caches the files for the experiment processes.

In [10]:
probe = L.LSIModel(BB.build_backbone(), C)
dev = np.load(os.path.join(C.DATA_DIR, "dev.npz"))
with torch.no_grad():
    logits, _ = probe(torch.from_numpy(dev["ids"][:2].astype(np.int64)), torch.from_numpy(dev["lens"][:2].astype(np.int64)))
print(f"model ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | logits {tuple(logits.shape)}")
del probe, dev

model_final.pt: reconstructing file:   0%|          |  0.00B /  200MB            

model_final.pt: downloading bytes:           |  0.00B            

Phase-1 arch 3 weights: /kaggle/working/phase5_arch3/phase1/d240357legalarch3decoder/model_final.pt (hf://D-240357/legal-arch3-decoder)
model ready: 100.3M parameters | logits (2, 100)


## 6. Run the experiments
Each experiment is its own process (`run_lsi.py --method ...`); with two GPUs, two run at once, so the six take about
three rounds of `TRAIN_MINUTES` + ~10 minutes. `kd` starts as soon as `full` (its teacher) is done.
Results, checkpoints and zips are pushed to `D-240357/legal-lsi-arch3` as they finish.

In [11]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_lsi.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

[gpu1] [lorma] train 42,750 docs | dev 10,181 | test 13,019 | chunks used per doc 2.72 of 6 | statutes per doc 3.70
[gpu0] [full] train 42,750 docs | dev 10,181 | test 13,019 | chunks used per doc 2.72 of 6 | statutes per doc 3.70
[gpu1] Phase-1 arch 3 weights: /kaggle/working/phase5_arch3/phase1/d240357legalarch3decoder/model_final.pt (cache of hf://D-240357/legal-arch3-decoder)
[gpu0] Phase-1 arch 3 weights: /kaggle/working/phase5_arch3/phase1/d240357legalarch3decoder/model_final.pt (cache of hf://D-240357/legal-arch3-decoder)
[gpu0] [full] all weights trainable | params 100.3M, trainable 100.35M | loss asl | budget 40 min
[gpu1] [lorma] LoRMA on 24 attention projections (r=8) | params 100.7M, trainable 0.65M | loss asl | budget 40 min
[gpu1] [lorma] step 150 | 2,700 docs seen | loss 5.9934 | dev-subset macro-F1 0.1363 micro-F1 0.2566 (thr 0.29) | 10% of budget
[gpu0] [full] step 150 | 2,700 docs seen | loss 6.0231 | dev-subset macro-F1 0.1190 micro-F1 0.2404 (thr 0.28) | 10% of budg

## 7. Results
Dev macro-F1 (the competition metric), what per-statute thresholds bought, micro-F1, subset accuracy, cost, and the hardest statutes.

In [12]:
import pandas as pd

rows, worst = [], {}
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    rows.append({"method": r["method"], "dev macro-F1": r["dev_macro_f1"], "global thr": r["dev_macro_f1_global_thr"],
                 "per-statute thr": r["dev_macro_f1_per_label_thr"], "micro-F1": r["dev_micro_f1"],
                 "subset acc": r["dev_subset_acc"], "statutes/test doc": r["test_labels_per_doc"],
                 "docs seen": r["docs_seen"], "params (M)": r["params_total"] / 1e6,
                 "trainable %": 100 * r["params_trainable"] / r["params_total"], "minutes": r["minutes"],
                 "peak GPU GB": r["peak_gpu_gb"], "dev infer s": r["dev_inference_seconds"]})
    worst[r["method"]] = [(w["label"], round(w["f1"], 2), w["dev_support"]) for w in r["worst_labels"][:6]]
table = pd.DataFrame(rows).sort_values("dev macro-F1", ascending=False).round(4)
display(table)
print("hardest statutes per method (id, dev F1, dev support):")
for m, w in worst.items():
    print(f"  {m:8s} {w}")

,method,dev macro-F1,global thr,per-statute thr,micro-F1,subset acc,statutes/test doc,docs seen,params (M),trainable %,minutes,peak GPU GB,dev infer s
4,partial,0.3785,0.3308,0.3785,0.4740,0.0335,5.1234,37620,100.3454,25.3380,52.6,2.33,310.0
1,full,0.3659,0.3178,0.3659,0.4639,0.0302,5.0921,26622,100.3454,100.0000,50.8,6.20,263.9
2,kd,0.3636,0.3131,0.3636,0.4576,0.0219,5.4974,29394,62.5875,100.0000,46.6,4.20,159.7
0,adapter,0.3574,0.3136,0.3574,0.4457,0.0252,5.6070,30312,102.7247,2.5622,51.2,3.89,277.5
3,lorma,0.3508,0.2979,0.3508,0.4460,0.0263,5.4155,26028,100.7387,0.6413,53.6,4.17,334.1
5,prompt,0.3222,0.2689,0.3222,0.4107,0.0121,6.2197,29700,100.3577,0.2641,50.6,3.59,276.8


hardest statutes per method (id, dev F1, dev support):
  adapter  [(2, 0.05, 98), (8, 0.07, 92), (0, 0.08, 86), (84, 0.1, 68), (18, 0.1, 100), (27, 0.1, 83)]
  full     [(2, 0.05, 98), (8, 0.06, 92), (0, 0.06, 86), (27, 0.08, 83), (99, 0.1, 176), (84, 0.11, 68)]
  kd       [(2, 0.04, 98), (8, 0.07, 92), (27, 0.07, 83), (0, 0.08, 86), (99, 0.1, 176), (18, 0.11, 100)]
  lorma    [(2, 0.05, 98), (0, 0.06, 86), (8, 0.06, 92), (84, 0.09, 68), (99, 0.1, 176), (41, 0.11, 205)]
  partial  [(2, 0.07, 98), (8, 0.07, 92), (0, 0.09, 86), (27, 0.11, 83), (84, 0.12, 68), (18, 0.12, 100)]
  prompt   [(2, 0.05, 98), (84, 0.07, 68), (8, 0.08, 92), (0, 0.08, 86), (99, 0.09, 176), (18, 0.1, 100)]


## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root holding a
JSON list of `{"id": ..., "pred": [statute ids]}` for all 13,019 test documents (unique, valid IDs). The best on dev is
also copied to `/kaggle/working/submission.zip`. Upload the zips exactly as they are.

In [13]:
best = table.iloc[0]["method"]
shutil.copy(glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0], "/kaggle/working/submission.zip")
print(f"best on dev: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)

best on dev: partial -> /kaggle/working/submission.zip
   /kaggle/working/phase5_arch3/runs/adapter/submission_arch3_adapter.zip
   /kaggle/working/phase5_arch3/runs/full/submission_arch3_full.zip
   /kaggle/working/phase5_arch3/runs/kd/submission_arch3_kd.zip
   /kaggle/working/phase5_arch3/runs/lorma/submission_arch3_lorma.zip
   /kaggle/working/phase5_arch3/runs/partial/submission_arch3_partial.zip
   /kaggle/working/phase5_arch3/runs/prompt/submission_arch3_prompt.zip
